# 02 · Neural network basics

Everything the Tacotron model uses, on toy data. Run top to bottom.

In [ ]:
import torch, torch.nn as nn
torch.manual_seed(0)

## 1. Tensors, shapes and broadcasting

A **tensor** is an n-dimensional grid of numbers. Its **shape** says how big each axis is. In EchoTTS a batch of mels is `(batch, 80, frames)`.

**Broadcasting** lets you combine different shapes: size-1 axes are stretched to match. **Analogy:** adding one row of prices to every row of a shopping table.

In [ ]:
x = torch.arange(6.).reshape(2, 3)   # (2, 3)
row = torch.tensor([10., 20., 30.])  # (3,)  -> treated as (1, 3)
col = torch.tensor([[100.], [200.]]) # (2, 1)
print(x.shape, '\n', x)
print('x + row:\n', x + row)
print('x + col:\n', x + col)
print('transpose:', x.T.shape, ' unsqueeze(0):', x.unsqueeze(0).shape, ' sum over axis 1:', x.sum(1))

## 2. Autograd by hand

For `y = (w·x + b)²` with x=2, w=3, b=1:  u = w·x + b = 7, y = 49.

Chain rule: dy/du = 2u = 14, du/dw = x = 2, du/db = 1. So **dy/dw = 28** and **dy/db = 14**.

Autograd records every operation in a graph during the forward pass, then walks it backwards multiplying local derivatives. **Analogy:** a receipt of every step, read in reverse to see how much each input mattered.

In [ ]:
x = torch.tensor(2.0)
w = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)
y = (w * x + b) ** 2
y.backward()
print('y =', y.item(), ' dy/dw =', w.grad.item(), ' dy/db =', b.grad.item())

## 3. A small network with a hand-written training loop

Goal: learn `y = sin(x)`. Each step is **forward** (predict) → **loss** (how wrong) → **backward** (gradients) → **optimizer step** (nudge weights downhill). `zero_grad()` is needed because PyTorch *adds* gradients up by default.

In [ ]:
xs = torch.linspace(-3, 3, 200).unsqueeze(1)  # (200, 1)
ys = torch.sin(xs)                             # (200, 1)
net = nn.Sequential(nn.Linear(1, 32), nn.Tanh(), nn.Linear(32, 1))
opt = torch.optim.Adam(net.parameters(), lr=1e-2)
for step in range(500):
    pred = net(xs)                          # forward
    loss = ((pred - ys) ** 2).mean()        # MSE loss
    opt.zero_grad()                         # clear old gradients
    loss.backward()                         # backward
    opt.step()                              # update
    if step % 100 == 0: print(step, round(loss.item(), 5))
print('final loss', round(loss.item(), 5))

## 4. Embedding, Conv1d and LSTM

These three layers make up the Tacotron encoder.

- **Embedding** `(B, T)` int IDs → `(B, T, D)` vectors. A lookup card per letter.
- **Conv1d** expects **channels first**: `(B, C_in, T)` → `(B, C_out, T')`. A sliding window that looks at a few neighbours.
  With `padding = kernel // 2` the length stays the same.
- **LSTM** (`batch_first=True`) `(B, T, D_in)` → `(B, T, H)`, ×2 if bidirectional. Reads in order and keeps a memory.

In [ ]:
B, T = 4, 12
ids = torch.randint(1, 30, (B, T))
emb = nn.Embedding(30, 16)
conv = nn.Conv1d(16, 32, kernel_size=5, padding=2)
lstm = nn.LSTM(32, 8, batch_first=True, bidirectional=True)
e = emb(ids);                 print('ids   ', tuple(ids.shape), '-> embedding', tuple(e.shape))
c = conv(e.transpose(1, 2));  print('conv in', tuple(e.transpose(1, 2).shape), '-> out', tuple(c.shape))
o, (h, cell) = lstm(c.transpose(1, 2))
print('lstm in', tuple(c.transpose(1, 2).shape), '-> out', tuple(o.shape), ' h', tuple(h.shape))

## 5. Packing padded sequences

Sentences have different lengths, so we pad them with zeros into one rectangle. An LSTM would happily read the padding as if it were real text. **Packing** tells it each true length so it stops on time. **Analogy:** handing the reader a bookmark that says where each line really ends.

In [ ]:
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
lengths = torch.tensor([12, 9, 5, 3])                  # sorted longest first
x = torch.randn(4, 12, 32)
for i, L in enumerate(lengths): x[i, L:] = 0           # zero padding
packed = pack_padded_sequence(x, lengths, batch_first=True)
out_packed, _ = lstm(packed)
out, out_lens = pad_packed_sequence(out_packed, batch_first=True, total_length=12)
print('out', tuple(out.shape), 'lengths', out_lens.tolist())
print('padded positions are exactly zero:', bool((out[3, 3:] == 0).all()))

## 6. Exercises

Solve each in the empty cell under it **before** opening the hidden answers cell at the end.

1. Make a tensor of shape `(3, 1)` and one of shape `(1, 4)`. Predict the shape of their sum, then check.
2. For `y = x³ + 2x` at `x = 2`, compute dy/dx by hand, then confirm with autograd.
3. A Conv1d with `in=80, out=512, kernel=5, padding=2` gets a mel of shape `(8, 80, 100)`. What is the output shape? How many parameters (weights + biases)?
4. A bidirectional LSTM with `hidden_size=256` gets `(2, 50, 512)`. What is the output shape?
5. Modify the training loop in section 3 to learn `y = x²` on `[-2, 2]` and get the loss below 0.01.

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3


In [ ]:
# Exercise 4


In [ ]:
# Exercise 5


### Answers (hidden — expand only after trying)

In [ ]:
# 1: (3, 4)
print((torch.zeros(3, 1) + torch.zeros(1, 4)).shape)
# 2: dy/dx = 3x^2 + 2 = 14
x = torch.tensor(2.0, requires_grad=True); (x**3 + 2*x).backward(); print(x.grad)
# 3: (8, 512, 100); params = 512*80*5 + 512 = 205,312
c = nn.Conv1d(80, 512, 5, padding=2); print(c(torch.randn(8, 80, 100)).shape, sum(p.numel() for p in c.parameters()))
# 4: (2, 50, 512) because 256 forward + 256 backward
print(nn.LSTM(512, 256, batch_first=True, bidirectional=True)(torch.randn(2, 50, 512))[0].shape)
# 5:
xs = torch.linspace(-2, 2, 200).unsqueeze(1); ys = xs ** 2
net = nn.Sequential(nn.Linear(1, 32), nn.Tanh(), nn.Linear(32, 1)); opt = torch.optim.Adam(net.parameters(), lr=1e-2)
for _ in range(1000):
    loss = ((net(xs) - ys) ** 2).mean(); opt.zero_grad(); loss.backward(); opt.step()
print('loss', loss.item())